# Model 1: Predictive Content of Congressional Trading

**Research Question:** Do congressional trades contain information that improves prediction of stock returns beyond publicly available market information?

**Methodology:**
- **Target:** Cumulative Abnormal Return (CAR) adjusted by Fama-French 3 factors
- **Approach:** Compare predictive accuracy of models with vs without congressional trading features
- **Evaluation:** Out-of-sample R² with expanding window (Campbell & Thompson, 2008)
- **Inference:** Clark-West (2007) test for nested model comparison

**Theoretical Framework:**
- Grossman-Stiglitz (1980): Informed traders with low information acquisition costs
- If congressional trades improve CAR prediction → evidence of private information

---

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ML - Standard sklearn models
from sklearn.linear_model import (
    LinearRegression, 
    RidgeCV, 
    LassoCV, 
    ElasticNetCV
)
from sklearn.ensemble import (
    RandomForestRegressor, 
    GradientBoostingRegressor
)
from sklearn.preprocessing import StandardScaler
from scipy import stats

# Plotting
import matplotlib.pyplot as plt

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Setup complete')

Setup complete


In [2]:
# Figure style (paper-ready, no titles)
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#333333',
    'axes.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'font.family': 'serif',
    'font.size': 10,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

# Colors
C_BASE = '#2E4057'      # Dark blue - Base model
C_AUGMENTED = '#C44536' # Red - Augmented model
C_NEUTRAL = '#8B8B8B'   # Gray

# Output
OUTPUT_DIR = 'outputs/Model_1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Data

In [3]:
# Load panel
panel = pd.read_parquet('data/prediction_bases/panel_final_stock_month.parquet')

# Temporal ordering
panel['month_dt'] = pd.to_datetime(panel['month'].astype(str))
panel = panel.sort_values(['month_dt', 'ticker'])

print(f'Observations: {len(panel):,}')
print(f'Period: {panel["month"].min()} to {panel["month"].max()}')
print(f'Unique stocks: {panel["ticker"].nunique():,}')
print(f'Unique months: {panel["month"].nunique()}')

Observations: 32,169
Period: 2012-07 to 2024-11
Unique stocks: 2,117
Unique months: 148


## 2. Target Variable

We use **Cumulative Abnormal Return (CAR)** adjusted by Fama-French 3 factors rather than raw returns.

**Rationale:** 
- Raw returns include systematic components (market, size, value) that are publicly known
- CAR isolates the idiosyncratic component — the part not explained by public factors
- If congressmen have private information, it should predict CAR, not systematic returns

In [4]:
TARGET = 'ret_future_car30_ff3'

# Winsorize to reduce outlier influence
lower, upper = panel[TARGET].quantile([0.01, 0.99])
panel[TARGET] = panel[TARGET].clip(lower=lower, upper=upper)

# Drop missing
panel = panel.dropna(subset=[TARGET])

print(f'Target: {TARGET}')
print(f'Mean:   {panel[TARGET].mean():.4f}')
print(f'Std:    {panel[TARGET].std():.4f}')
print(f'Range:  [{panel[TARGET].min():.4f}, {panel[TARGET].max():.4f}]')
print(f'N:      {len(panel):,}')

Target: ret_future_car30_ff3
Mean:   -0.0014
Std:    0.0871
Range:  [-0.2592, 0.2670]
N:      31,943


## 3. Feature Sets

We define two feature sets:
1. **Base (Market):** Public information available to all investors
2. **Augmented (Market + Congress):** Base plus congressional trading features

The comparison tests whether congressional features add predictive content.

In [5]:
# MARKET FEATURES (Public Information)
# Based on standard asset pricing literature
FEATURES_MARKET = [
    # Momentum (Jegadeesh & Titman, 1993)
    'mkt_momentum_20d',
    'mkt_momentum_60d', 
    'mkt_momentum_252d',
    
    # Volatility
    'mkt_realized_vol_30d',
    'mkt_realized_vol_252d',
    
    # Liquidity (Amihud, 2002)
    'mkt_amihud_illiq_20d',
    'mkt_volume_ratio_30d',
    
    # Factor exposures (Fama & French, 1993)
    'mkt_beta_252d',
    'mkt_beta_smb_ff3_252d',
    'mkt_beta_hml_ff3_252d',
    
    # Valuation
    'mkt_price_to_book',
    'mkt_market_cap',
]

# Filter to available columns
FEATURES_MARKET = [f for f in FEATURES_MARKET if f in panel.columns]
print(f'Market features: {len(FEATURES_MARKET)}')

Market features: 12


In [6]:
# CONGRESSIONAL FEATURES (Our contribution)
# Selected based on theoretical relevance to information hypothesis
FEATURES_CONGRESS = [
    # Direction of trading
    'cong_csi',              # Congressional Sentiment Index
    'cong_buy_ratio',        # Proportion of buys
    
    # Information access proxies
    'cong_info_ratio',       # % from informative committees
    'cong_chair_ratio',      # % from chairs/ranking members
    
    # Political power
    'cong_avg_power_index',  # Composite power measure
    'cong_senator_ratio',    # % from senators
    
    # Disclosure timing
    'cong_avg_disclosure_delay',
    'cong_long_delay_ratio',
    
    # Coordination
    'cong_coordinated_ratio',
    'cong_unique_politicians',
    
    # Composite signals
    'cong_smart_money',
    'cong_strong_buy',
]

# Filter to available columns
FEATURES_CONGRESS = [f for f in FEATURES_CONGRESS if f in panel.columns]
print(f'Congressional features: {len(FEATURES_CONGRESS)}')

Congressional features: 12


In [7]:
# Combined set
FEATURES_ALL = FEATURES_MARKET + FEATURES_CONGRESS

print(f'\nFeature sets:')
print(f'  Base (Market only):     {len(FEATURES_MARKET)} features')
print(f'  Augmented (+ Congress): {len(FEATURES_ALL)} features')


Feature sets:
  Base (Market only):     12 features
  Augmented (+ Congress): 24 features


In [9]:
# Handle missing values in features
for col in FEATURES_ALL:
    if panel[col].isna().any():
        panel[col] = panel[col].fillna(panel[col].median())

# Winsorize features
for col in FEATURES_ALL:
    lower, upper = panel[col].quantile([0.01, 0.99])
    panel[col] = panel[col].clip(lower=lower, upper=upper)

print(f'Missing values after cleaning: {panel[FEATURES_ALL].isna().sum().sum()}')

Missing values after cleaning: 0


## 4. Evaluation Framework

**Out-of-sample R² (Campbell & Thompson, 2008):**
$$R^2_{OOS} = 1 - \frac{\sum_t (r_t - \hat{r}_t)^2}{\sum_t (r_t - \bar{r}_t)^2}$$

where $\bar{r}_t$ is the expanding window historical mean (benchmark).

**Clark-West (2007) test:** For nested models, tests whether the unrestricted model significantly outperforms.

In [10]:
def oos_r2(y_true, y_pred, y_benchmark):
    """Out-of-sample R² (Campbell & Thompson, 2008)."""
    sse_model = np.sum((y_true - y_pred) ** 2)
    sse_bench = np.sum((y_true - y_benchmark) ** 2)
    if sse_bench == 0:
        return np.nan
    return 1 - (sse_model / sse_bench)


def clark_west_test(y_true, pred_restricted, pred_unrestricted):
    """
    Clark-West (2007) test for nested models.
    H0: Restricted model is sufficient
    H1: Unrestricted model improves prediction
    """
    e1 = y_true - pred_restricted
    e2 = y_true - pred_unrestricted
    adj = (pred_restricted - pred_unrestricted) ** 2
    f = e1**2 - (e2**2 - adj)
    
    t_stat = np.mean(f) / (np.std(f, ddof=1) / np.sqrt(len(f)))
    p_value = 1 - stats.norm.cdf(t_stat)
    return t_stat, p_value

In [11]:
class ExpandingWindowCV:
    """
    Expanding window cross-validation for panel data.
    
    At each period t:
    - Train on all observations from periods [1, t-1]
    - Predict all observations in period t
    - No future information is used (prevents look-ahead bias)
    """
    
    def __init__(self, min_train_periods=36):
        self.min_train = min_train_periods
    
    def generate_predictions(self, data, target, features, model):
        """Generate out-of-sample predictions."""
        months = sorted(data['month_dt'].unique())
        
        if len(months) <= self.min_train:
            return np.array([]), np.array([])
        
        all_preds = []
        all_actuals = []
        
        for t in range(self.min_train, len(months)):
            # Split
            train_mask = data['month_dt'].isin(months[:t])
            test_mask = data['month_dt'] == months[t]
            
            X_train = data.loc[train_mask, features].values
            y_train = data.loc[train_mask, target].values
            X_test = data.loc[test_mask, features].values
            y_test = data.loc[test_mask, target].values
            
            if len(X_test) == 0:
                continue
            
            # Scale (fit on train only)
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            # Fit and predict
            model.fit(X_train_scaled, y_train)
            preds = model.predict(X_test_scaled)
            
            all_preds.extend(preds)
            all_actuals.extend(y_test)
        
        return np.array(all_preds), np.array(all_actuals)
    
    def historical_mean_benchmark(self, data, target):
        """Expanding window historical mean."""
        months = sorted(data['month_dt'].unique())
        
        all_preds = []
        all_actuals = []
        
        for t in range(self.min_train, len(months)):
            train_mask = data['month_dt'].isin(months[:t])
            test_mask = data['month_dt'] == months[t]
            
            hist_mean = data.loc[train_mask, target].mean()
            y_test = data.loc[test_mask, target].values
            
            all_preds.extend([hist_mean] * len(y_test))
            all_actuals.extend(y_test)
        
        return np.array(all_preds), np.array(all_actuals)

## 5. Models

We use standard models from the ML literature with regularization to prevent overfitting:
- **OLS:** Baseline linear model
- **Ridge:** L2 regularization (shrinks coefficients)
- **LASSO:** L1 regularization (sparse feature selection)
- **Elastic Net:** Combines L1 and L2
- **Random Forest:** Non-linear, with depth constraints
- **Gradient Boosting:** Sequential ensemble, conservative parameters

In [12]:
MODELS = {
    'OLS': LinearRegression(),
    
    'Ridge': RidgeCV(
        alphas=np.logspace(-3, 3, 30),
        cv=5
    ),
    
    'LASSO': LassoCV(
        alphas=np.logspace(-4, 0, 30),
        cv=5,
        max_iter=5000,
        random_state=RANDOM_STATE
    ),
    
    'ElasticNet': ElasticNetCV(
        l1_ratio=[0.1, 0.5, 0.9],
        alphas=np.logspace(-4, 0, 20),
        cv=5,
        max_iter=5000,
        random_state=RANDOM_STATE
    ),
    
    'RandomForest': RandomForestRegressor(
        n_estimators=100,
        max_depth=3,
        min_samples_leaf=50,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=100,
        max_depth=2,
        learning_rate=0.01,
        min_samples_leaf=50,
        subsample=0.8,
        random_state=RANDOM_STATE
    ),
}

print(f'Models: {list(MODELS.keys())}')

Models: ['OLS', 'Ridge', 'LASSO', 'ElasticNet', 'RandomForest', 'GradientBoosting']


## 6. Model Comparison

In [13]:
# Initialize CV
cv = ExpandingWindowCV(min_train_periods=36)

# Benchmark
print('Computing benchmark (historical mean)...')
bench_pred, bench_actual = cv.historical_mean_benchmark(panel, TARGET)
print(f'Out-of-sample predictions: {len(bench_pred):,}')

Computing benchmark (historical mean)...
Out-of-sample predictions: 29,442


In [14]:
# Feature sets to compare
FEATURE_SETS = {
    'Base': FEATURES_MARKET,
    'Augmented': FEATURES_ALL,
}

# Store results
results = []
predictions = {'benchmark': bench_pred}

print('\nRunning models...')
print('='*60)

for feat_name, features in FEATURE_SETS.items():
    print(f'\n{feat_name} ({len(features)} features):')
    
    for model_name, model in MODELS.items():
        print(f'  {model_name}...', end=' ')
        
        # Generate predictions
        pred, actual = cv.generate_predictions(panel, TARGET, features, model)
        
        if len(pred) == 0:
            print('skipped')
            continue
        
        # Align lengths
        n = min(len(pred), len(bench_pred))
        pred, actual, bench = pred[:n], actual[:n], bench_pred[:n]
        
        # Store predictions
        predictions[f'{model_name}_{feat_name}'] = pred
        
        # Compute metrics
        r2 = oos_r2(actual, pred, bench)
        mse = np.mean((actual - pred) ** 2)
        corr = np.corrcoef(actual, pred)[0, 1]
        
        results.append({
            'Model': model_name,
            'Features': feat_name,
            'N_features': len(features),
            'N_obs': len(pred),
            'R2_OOS': r2,
            'R2_OOS_pct': r2 * 100,
            'MSE': mse,
            'Correlation': corr,
        })
        
        print(f'R²={r2*100:+.3f}%')

results_df = pd.DataFrame(results)


Running models...

Base (12 features):
  OLS... R²=-0.027%
  Ridge... R²=+0.043%
  LASSO... R²=+0.065%
  ElasticNet... R²=+0.055%
  RandomForest... R²=+0.325%
  GradientBoosting... R²=+0.424%

Augmented (24 features):
  OLS... R²=-0.103%
  Ridge... R²=-0.002%
  LASSO... R²=+0.052%
  ElasticNet... R²=+0.055%
  RandomForest... R²=+0.321%
  GradientBoosting... R²=+0.405%


## 7. Results

In [ ]:
# Pivot table
pivot = results_df.pivot(index='Model', columns='Features', values='R2_OOS_pct')
pivot['Δ (pp)'] = pivot['Augmented'] - pivot['Base']
pivot = pivot.sort_values('Δ (pp)', ascending=False)

print('OUT-OF-SAMPLE R² (%)')
print('='*50)
print(pivot.round(4).to_string())

In [ ]:
# Clark-West tests
print('\nCLARK-WEST TEST')
print('H0: Base model sufficient | H1: Augmented improves')
print('='*50)

cw_results = []
actual = bench_actual[:len(bench_pred)]

for model_name in MODELS.keys():
    key_base = f'{model_name}_Base'
    key_aug = f'{model_name}_Augmented'
    
    if key_base in predictions and key_aug in predictions:
        pred_base = predictions[key_base]
        pred_aug = predictions[key_aug]
        
        n = min(len(pred_base), len(pred_aug), len(actual))
        cw_stat, cw_pval = clark_west_test(
            actual[:n], pred_base[:n], pred_aug[:n]
        )
        
        sig = '***' if cw_pval < 0.01 else '**' if cw_pval < 0.05 else '*' if cw_pval < 0.10 else ''
        
        cw_results.append({
            'Model': model_name,
            'CW_stat': cw_stat,
            'p_value': cw_pval,
        })
        
        print(f'{model_name:18s}  CW={cw_stat:+6.3f}  p={cw_pval:.4f} {sig}')

cw_df = pd.DataFrame(cw_results)

## 8. Figures

In [ ]:
# Figure 1: R² comparison by model
fig, ax = plt.subplots(figsize=(8, 4.5))

models = pivot.index.tolist()
x = np.arange(len(models))
width = 0.35

ax.bar(x - width/2, pivot['Base'], width, label='Base (Market)', 
       color=C_BASE, edgecolor='white', linewidth=0.5)
ax.bar(x + width/2, pivot['Augmented'], width, label='Augmented (+Congress)', 
       color=C_AUGMENTED, edgecolor='white', linewidth=0.5)

ax.axhline(0, color=C_NEUTRAL, linewidth=0.8)
ax.set_ylabel('Out-of-Sample R² (%)')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha='right')
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig1_r2_comparison.pdf')
plt.savefig(f'{OUTPUT_DIR}/fig1_r2_comparison.png')
plt.show()

In [ ]:
# Figure 2: Improvement from congressional features
fig, ax = plt.subplots(figsize=(6, 4))

improvements = pivot['Δ (pp)'].sort_values()
colors = [C_AUGMENTED if v > 0 else C_NEUTRAL for v in improvements]

ax.barh(improvements.index, improvements.values, color=colors, 
        edgecolor='white', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Δ R² OOS (percentage points)')

for i, (model, val) in enumerate(improvements.items()):
    x_pos = val + 0.003 if val >= 0 else val - 0.003
    ha = 'left' if val >= 0 else 'right'
    ax.text(x_pos, i, f'{val:+.3f}', va='center', ha=ha, fontsize=8)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig2_improvement.pdf')
plt.savefig(f'{OUTPUT_DIR}/fig2_improvement.png')
plt.show()

## 9. Export

In [ ]:
# Save results
results_df.to_csv(f'{OUTPUT_DIR}/results_full.csv', index=False)
pivot.to_csv(f'{OUTPUT_DIR}/results_pivot.csv')
cw_df.to_csv(f'{OUTPUT_DIR}/clark_west_tests.csv', index=False)

print(f'Results saved to {OUTPUT_DIR}/')

## 10. Summary

In [ ]:
# Summary statistics
n_improve = (pivot['Δ (pp)'] > 0).sum()
n_total = len(pivot)
avg_improve = pivot['Δ (pp)'].mean()
n_sig = (cw_df['p_value'] < 0.10).sum()

best_base = results_df[results_df['Features'] == 'Base'].sort_values('R2_OOS', ascending=False).iloc[0]
best_aug = results_df[results_df['Features'] == 'Augmented'].sort_values('R2_OOS', ascending=False).iloc[0]

print('='*60)
print('SUMMARY')
print('='*60)
print(f'''
Data:
  Observations:      {len(panel):,}
  OOS predictions:   {len(bench_pred):,}
  Target:            {TARGET}
  Market features:   {len(FEATURES_MARKET)}
  Congress features: {len(FEATURES_CONGRESS)}

Best Base Model:
  {best_base['Model']}: R²_OOS = {best_base['R2_OOS_pct']:.4f}%

Best Augmented Model:
  {best_aug['Model']}: R²_OOS = {best_aug['R2_OOS_pct']:.4f}%

Comparison:
  Models improved:     {n_improve}/{n_total}
  Avg improvement:     {avg_improve:+.4f} pp
  Significant (p<0.1): {n_sig}/{n_total}
''')

if avg_improve > 0 and n_improve > n_total/2:
    print('Conclusion: Congressional features improve prediction of abnormal returns.')
elif avg_improve > 0:
    print('Conclusion: Mixed evidence - some models improve, others do not.')
else:
    print('Conclusion: No evidence that congressional features improve prediction.')

print('='*60)